# Lesson 7.4 - AI Product & System Design (architecture notebook)

## Objectives

- Translate product goals into AI system architecture.
- Document non-functional requirements (latency, cost, reliability, security).
- Build a lightweight code skeleton for a production-minded AI service.


## Product Brief Template

Use this structure for any AI-native product proposal:

1. Problem statement
2. User segments
3. Baseline process and pain points
4. AI leverage points
5. Risk and governance constraints
6. KPIs and success criteria


## Architecture Sketch (Text Diagram)

```text
Client UI
  -> API Gateway
    -> Orchestrator
      -> Retrieval Service
      -> Agent Runtime
      -> Tool Gateway
    -> Response Formatter
  -> Observability (logs/metrics/traces)
  -> Data Stores (vector, relational, event)
```


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("ai_system")


In [ ]:
@dataclass
class RequestContext:
    user_id: str
    tenant_id: str
    intent: str
    risk_level: str


class RetrievalService:
    def fetch(self, query: str) -> str:
        return f"Retrieved context for: {query}"


class AgentRuntime:
    def run(self, prompt: str, context: str) -> str:
        return f"Agent output based on [{prompt}] with [{context}]"


class PolicyGuard:
    def check(self, ctx: RequestContext, output: str) -> bool:
        if ctx.risk_level == "high" and "approve" not in output.lower():
            return False
        return True


class AIOrchestrator:
    def __init__(self) -> None:
        self.retriever = RetrievalService()
        self.agent = AgentRuntime()
        self.guard = PolicyGuard()

    def handle(self, ctx: RequestContext, user_prompt: str) -> Dict[str, Any]:
        retrieved = self.retriever.fetch(user_prompt)
        output = self.agent.run(user_prompt, retrieved)
        allowed = self.guard.check(ctx, output)
        logger.info("request=%s allowed=%s", ctx.user_id, allowed)
        return {
            "retrieved": retrieved,
            "output": output,
            "allowed": allowed,
            "requires_human_review": not allowed,
        }


orchestrator = AIOrchestrator()
result = orchestrator.handle(
    RequestContext(user_id="u-1", tenant_id="t-1", intent="policy_change", risk_level="high"),
    "Draft an approval-ready policy update summary",
)
result


## Example API Contract (YAML)

```yaml
POST /v1/agent/tasks
request:
  user_id: string
  tenant_id: string
  goal: string
  risk_level: low|medium|high
response:
  task_id: string
  status: accepted|requires_review
  preview: string
  trace_id: string
```


## Configuration Snippet (JSON)

```json
{
  "routing": {
    "low_risk_model": "small-instruct-model",
    "high_risk_model": "strong-reasoning-model"
  },
  "limits": {
    "max_tool_calls": 8,
    "max_context_tokens": 12000,
    "timeout_seconds": 20
  },
  "governance": {
    "human_review_for_high_risk": true,
    "pii_redaction": true
  }
}
```


## Business Case Studies & Exceptions

### Case Study A: Customer Support Agentic Platform

A support product shipped with explicit routing and policy checks. The team started with a modular monolith and only split services once p95 latency and ownership boundaries justified it.

### Case Study B: Data Leak Incident

A prototype stored raw prompts with PII in logs. Governance remediation required masked logging, tenant-level key isolation, and retention controls.

### Exceptions

- In heavily regulated environments, external model APIs may be disallowed.
- Some workflows need deterministic rules first, with AI suggestions only.


## Interview Questions & Answers

1. **What is AI product design?**  
   Designing end-to-end user and business outcomes, not just model outputs.
2. **What comes before model choice?**  
   Problem framing, user journey, risk profile, and success metrics.
3. **How do you set AI system SLOs?**  
   Define latency, success rate, cost, and reliability targets tied to user value.
4. **When is a modular monolith preferred?**  
   Early stages when speed and operational simplicity matter most.
5. **How do you enforce safe execution?**  
   Policy guardrails, scoped tools, and human approval for sensitive actions.
6. **How do you prevent cost blow-ups?**  
   Token budgets, caching, routing, and observability.
7. **What is a trace_id for?**  
   End-to-end debugging and auditability across components.
8. **Why include a tool gateway?**  
   To centralize auth, rate limits, and policy enforcement.
9. **How do you choose between API-hosted and self-hosted models?**  
   Trade off control/privacy against speed of delivery and operational burden.
10. **How do you evaluate production readiness?**  
   Verify reliability, security, governance, and rollback pathways.
